# Notebook 4: Daten Transformieren - T in ETL

## Lernziele

Nach diesem Notebook kannst du:
- Datentypen erkennen und korrigieren
- Fehlende Werte gezielt behandeln
- Duplikate finden und entfernen
- Textspalten bereinigen (strip, lower, replace)
- Neue Spalten aus bestehenden berechnen
- Daten aus verschiedenen Quellen zusammenführen (merge/join)
- Daten pivotieren und umstrukturieren

---

## Warum ist Transform der wichtigste Schritt?

"Garbage in, garbage out" - wenn die Eingabedaten schlecht sind, liefert jede Analyse falsche Ergebnisse. In der Praxis verbringen Datenprofis **60–80% der Zeit** mit Datentransformation und -bereinigung.

In [ ]:
import pandas as pd
import numpy as np

print("Bibliotheken geladen")

---
# 1. Der "schmutzige" Datensatz

Wir arbeiten mit einem realistisch schmutzigen Verkaufsdatensatz - mit absichtlich eingebauten typischen Problemen.

In [ ]:
rohdaten = [
    {"bestellung_id": "B-001", "datum": "15.01.2024", "kunde": "  ANNA SCHMIDT ",
     "produkt": "Laptop Pro", "menge": "2", "preis": "999,99", "region": "nord", "status": "geliefert"},
    {"bestellung_id": "B-002", "datum": "15.01.2024", "kunde": "Klaus Müller",
     "produkt": "USB-Maus", "menge": "5", "preis": "29,90", "region": "SÜD", "status": "geliefert"},
    {"bestellung_id": "B-003", "datum": "16/01/2024", "kunde": "Sara Wagner",
     "produkt": "Monitor", "menge": None, "preis": "349.00", "region": "Ost", "status": "versandt"},
    {"bestellung_id": "B-004", "datum": "16.01.2024", "kunde": "  Tom Becker  ",
     "produkt": "tastatur", "menge": "3", "preis": "N/A", "region": "NORD", "status": "offen"},
    {"bestellung_id": "B-005", "datum": "17.01.2024", "kunde": "Lisa Braun",
     "produkt": "Webcam HD", "menge": "2", "preis": "79,00", "region": "süd", "status": "geliefert"},
    {"bestellung_id": "B-002", "datum": "15.01.2024", "kunde": "Klaus Müller",  # DUPLIKAT!
     "produkt": "USB-Maus", "menge": "5", "preis": "29,90", "region": "SÜD", "status": "geliefert"},
    {"bestellung_id": "B-006", "datum": "18.01.2024", "kunde": "Marc Schäfer",
     "produkt": "Headset", "menge": "-1", "preis": "89,00", "region": "ost", "status": "storniert"},
    {"bestellung_id": "B-007", "datum": "18.01.2024", "kunde": None,
     "produkt": "SSD 1TB", "menge": "1", "preis": "129,90", "region": "nord", "status": "offen"},
]

df = pd.DataFrame(rohdaten)
print(f"Shape: {df.shape}")
df

---
# 2. Daten inspizieren - was ist alles falsch?

Schritt 1 der Transformation ist immer eine gründliche Inspektion.

In [ ]:
print("=== Datentypen ===")
print(df.dtypes)

print("\n=== Fehlende Werte ===")
print(df.isnull().sum())

print("\n=== Duplikate ===")
print(f"Duplikate: {df.duplicated().sum()}")
print(f"Duplikate (nach bestellung_id): {df.duplicated(subset=['bestellung_id']).sum()}")

print("\n=== Einzigartige Werte in Textspalten ===")
for col in ["region", "status"]:
    print(f"  {col}: {sorted(df[col].unique())}")

**Erkannte Probleme:**
1. `datum` - verschiedene Formate (`.`, `/`), als String gespeichert
2. `preis` - Komma statt Punkt, "N/A" als Text, String statt float
3. `menge` - String statt int, `None`, negativer Wert (-1)
4. `kunde` - Leerzeichen, Großschreibung, fehlender Wert
5. `produkt` - Großschreibung inkonsistent
6. `region` - verschiedene Schreibweisen (NORD, nord, Ost, ost)
7. Duplikat (B-002 doppelt)

---
# 3. Schritt-für-Schritt Transformation

In [ ]:
# Wir arbeiten auf einer Kopie - rohdaten bleiben unverändert
df_clean = df.copy()

# ==== 3.1 Duplikate entfernen ====
vor = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["bestellung_id"], keep="first")
print(f"Duplikate entfernt: {vor - len(df_clean)} Zeilen")
print(f"Zeilen jetzt: {len(df_clean)}")

### Der `.str`-Accessor

Für Textspalten stellt pandas den **`.str`-Accessor** bereit. Er ermöglicht String-Methoden direkt auf ganzen Spalten anzuwenden - ohne Schleife.

| Methode | Wirkung | Beispiel |
|---------|---------|---------|
| `.str.strip()` | Leerzeichen am Rand entfernen | `"  Anna  "` → `"Anna"` |
| `.str.lower()` | Alles klein | `"NORD"` → `"nord"` |
| `.str.title()` | Erster Buchstabe groß | `"anna schmidt"` → `"Anna Schmidt"` |
| `.str.replace(alt, neu)` | Zeichen ersetzen | `"999,99"` → `"999.99"` |

Fehlende Werte (`NaN`) werden dabei automatisch übersprungen.

## 3.2 Textspalten bereinigen

In [ ]:

# Kunde: Leerzeichen entfernen, korrekte Großschreibung (title case)
df_clean["kunde"] = df_clean["kunde"].str.strip()
df_clean["kunde"] = df_clean["kunde"].str.title()

# Produkt: Konsistente Großschreibung
df_clean["produkt"] = df_clean["produkt"].str.strip()
df_clean["produkt"] = df_clean["produkt"].str.title()

# Region: Einheitlich klein
df_clean["region"] = df_clean["region"].str.strip()
df_clean["region"] = df_clean["region"].str.lower()

# Status: Einheitlich klein
df_clean["status"] = df_clean["status"].str.strip()
df_clean["status"] = df_clean["status"].str.lower()

print("Textspalten bereinigt.")
print(f"Regionen jetzt: {sorted(df_clean['region'].dropna().unique())}")
print(f"Status jetzt:   {sorted(df_clean['status'].dropna().unique())}")

## 3.3 Preis bereinigen und in Zahl umwandeln

In [ ]:
def preis_zu_float(wert):
    """Wandelt einen Preis-String in float um. Ungültige Werte ergeben 0.0."""
    if wert in ("N/A", "", None, "-"):
        return 0.0
    wert_str = str(wert)
    wert_str = wert_str.replace(",", ".")
    return float(wert_str)

df_clean["preis"] = df_clean["preis"].apply(preis_zu_float)
print(f"Preise: {df_clean['preis'].tolist()}")
print(f"Typ: {df_clean['preis'].dtype}")

## 3.4 Menge bereinigen

In [ ]:
# In int umwandeln (fehlende -> 0): drei Schritte
df_clean["menge"] = pd.to_numeric(df_clean["menge"], errors="coerce")  # Strings -> Zahl (ungültig -> NaN)
df_clean["menge"] = df_clean["menge"].fillna(0)                         # NaN -> 0
df_clean["menge"] = df_clean["menge"].astype(int)                       # float -> int

# Negative Mengen auf 0 setzen (logisch ungültig)
ungueltige_menge = (df_clean["menge"] < 0).sum()
# .loc[zeilen_bedingung, spaltenname] = wert
# ermöglicht gezieltes Überschreiben von Werten in gefilterten Zeilen
df_clean.loc[df_clean["menge"] < 0, "menge"] = 0
print(f"Negative Mengen korrigiert: {ungueltige_menge}")

## 3.5 Datum parsen (mehrere Formate)

In [ ]:
# dayfirst=True für deutsches Format (TT.MM.JJJJ)
df_clean["datum"] = pd.to_datetime(df_clean["datum"], dayfirst=True, errors="coerce")

print(f"Datum-Typ: {df_clean['datum'].dtype}")
print(f"Datum ungültig: {df_clean['datum'].isnull().sum()}")

## 3.6 Fehlende Werte behandeln 

In [ ]:
print("Fehlende Werte nach Bereinigung:")
print(df_clean.isnull().sum())

# Fehlende Kunden: Zeilen kennzeichnen statt löschen
df_clean["kunde"] = df_clean["kunde"].fillna("Unbekannt")

# Fehlender Preis: Beibehalten (als NaN) - wird später bei Berechnungen ausgeschlossen
# (Alternative: df_clean["preis"].fillna(0) oder df_clean.dropna(subset=["preis"]))

### Der `.dt`-Accessor

Für Spalten mit Datums- oder Zeitwerten (`datetime64`) bietet pandas den **`.dt`-Accessor** - ähnlich wie `.str` für Texte.

| Attribut / Methode | Wirkung |
|--------------------|---------|
| `.dt.year`, `.dt.month`, `.dt.day` | Jahr, Monat, Tag als Zahl |
| `.dt.day_name()` | Wochentag als Text (`"Monday"`, ...) |
| `.dt.isocalendar().week` | Kalenderwoche |
| `.dt.days` | Differenz in Tagen (bei `Timedelta`-Spalten) |

Voraussetzung: Die Spalte muss vorher mit `pd.to_datetime()` umgewandelt worden sein.

## 3.7 Neue berechnete Spalten 

In [ ]:
# Gesamtwert der Bestellung
df_clean["gesamtwert"] = df_clean["preis"] * df_clean["menge"]

# Zeitbasierte Spalten aus dem Datum
df_clean["wochentag"] = df_clean["datum"].dt.day_name(locale=None)  # Englisch
df_clean["kalenderwoche"] = df_clean["datum"].dt.isocalendar().week

# Kategorisierung
df_clean["ist_abgeschlossen"] = df_clean["status"] == "geliefert"

print("Neue Spalten hinzugefügt:")
print(df_clean[["bestellung_id", "gesamtwert", "wochentag", "ist_abgeschlossen"]].head())

In [ ]:
# Endergebnis
print("=== Bereinigter DataFrame ===")
df_clean

In [ ]:
# Vorher/Nachher Vergleich
print("VORHER (Rohdaten):")
display(df.dtypes)
print(f"Fehlend: {df.isnull().sum().sum()}, Duplikate: {df.duplicated().sum()}")

print("\nNACHHER (Bereinigt):")
display(df_clean.dtypes)
print(f"Fehlend: {df_clean.isnull().sum().sum()}, Duplikate: {df_clean.duplicated().sum()}")

---
### Übung 1: Transformations-Pipeline

Du bekommst einen neuen schmutzigen Datensatz. Bereinige ihn vollständig.

Probleme (die du finden und lösen musst):
- Inkonsistente Texte
- Falsche Datentypen
- Fehlende Werte
- Ungültige Werte

> **Übung → siehe [`04_ETL_Transform_Aufgaben.ipynb`](04_ETL_Transform_Aufgaben.ipynb)**

---
# 4. Fortgeschrittene Transformationen

## 4.1 apply() - benutzerdefinierte Transformation

In [ ]:
# apply() wendet eine Funktion auf jede Zeile oder Spalte an

def risiko_kategorie(row):
    """Bewertet das Risiko einer Bestellung."""
    if row["status"] == "storniert":
        return "Storno"
    elif pd.isna(row["preis"]) or row["menge"] == 0:
        return "Datenproblem"
    elif row["gesamtwert"] > 500:
        return "Hochwertig"
    else:
        return "Normal"

# axis=1 bedeutet: Funktion auf jede ZEILE anwenden
df_clean["risiko"] = df_clean.apply(risiko_kategorie, axis=1)
print(df_clean[["bestellung_id", "gesamtwert", "status", "risiko"]])

## 4.2 Pivot-Tabellen

Pivot-Tabellen transformieren Zeilen in Spalten - ideal für Berichte.

In [ ]:
# Beispiel: Umsatz pro Monat und Region
umsatz_df = pd.DataFrame([
    {"monat": "Jan", "region": "nord", "umsatz": 45000},
    {"monat": "Jan", "region": "süd",  "umsatz": 32000},
    {"monat": "Jan", "region": "ost",  "umsatz": 28000},
    {"monat": "Feb", "region": "nord", "umsatz": 51000},
    {"monat": "Feb", "region": "süd",  "umsatz": 38000},
    {"monat": "Feb", "region": "ost",  "umsatz": 31000},
    {"monat": "Mär", "region": "nord", "umsatz": 49000},
    {"monat": "Mär", "region": "süd",  "umsatz": 41000},
    {"monat": "Mär", "region": "ost",  "umsatz": 35000},
])

print("Original (Long Format):")
display(umsatz_df)

In [ ]:
# Pivot: Zeilen = Monate, Spalten = Regionen
pivot = umsatz_df.pivot_table(
    index="monat",        # Was kommt in die Zeilen?
    columns="region",     # Was kommt in die Spalten?
    values="umsatz",      # Welcher Wert wird aggregiert?
    aggfunc="sum"         # Wie aggregieren? (sum, mean, count, ...)
)

print("Pivotiert (Wide Format):")
display(pivot)

# Gesamtsummen hinzufügen
pivot["Gesamt"] = pivot.sum(axis=1)
pivot.loc["Gesamt"] = pivot.sum()
pivot

## 4.3 Daten zusammenfügen (Merge/Join)

Im ETL-Kontext werden häufig Daten aus verschiedenen Quellen zusammengeführt.

In [ ]:
# Bestellungen aus Transaktionssystem
bestellungen = pd.DataFrame({
    "bestellung_id": ["B001", "B002", "B003", "B004", "B005"],
    "kunden_id":     [101, 102, 101, 103, 104],
    "betrag":        [999, 30, 350, 80, 150]
})

# Kundendaten aus CRM
kunden = pd.DataFrame({
    "kunden_id": [101, 102, 103],
    "name":      ["Anna Schmidt", "Klaus Müller", "Sara Wagner"],
    "segment":   ["Premium", "Standard", "Premium"]
})

print("Bestellungen:")
display(bestellungen)
print("Kunden:")
display(kunden)

In [ ]:
# LEFT JOIN: Alle Bestellungen behalten, Kundendaten ergänzen wo vorhanden
# (B005 hat kunden_id 104 - nicht in Kundentabelle -> NaN)
bestellungen_mit_kunden = pd.merge(
    bestellungen,   # linke Tabelle (alle Zeilen bleiben)
    kunden,         # rechte Tabelle
    on="kunden_id",
    how="left"      # 'left', 'right', 'inner', 'outer'
)
bestellungen_mit_kunden

In [ ]:
# JOIN-Typen erklärt:
print("inner: Nur Zeilen mit Übereinstimmung in beiden Tabellen")
display(pd.merge(bestellungen, kunden, on="kunden_id", how="inner"))

print("\nleft: Alle Zeilen der linken Tabelle, rechts ergänzt wo vorhanden")
display(pd.merge(bestellungen, kunden, on="kunden_id", how="left"))

---
### Übung 2: Vollständige Transformations-Pipeline

Baue eine Funktion `daten_transformieren(df)`, die folgende Schritte auf den `df_clean` DataFrame anwendet:

1. Nur Bestellungen mit Status "geliefert" oder "versandt" behalten (aktive Bestellungen)
2. Eine neue Spalte `mehrwertsteuer` berechnen (19% des Gesamtwerts)
3. Eine neue Spalte `gesamtwert_brutto` berechnen (Gesamtwert + MwSt)
4. Den DataFrame nach `gesamtwert_brutto` absteigend sortieren
5. Nur die Spalten `bestellung_id`, `datum`, `kunde`, `produkt`, `gesamtwert_brutto`, `region` zurückgeben

Gib den transformierten DataFrame zurück und zeige ihn an.

> **Übung → siehe [`04_ETL_Transform_Aufgaben.ipynb`](04_ETL_Transform_Aufgaben.ipynb)**

---
## Zusammenfassung

In diesem Notebook hast du gelernt, Rohdaten zu bereinigen und umzuformen:

| Schritt | Methoden |
|---------|---------|
| **Inspizieren** | `dtypes`, `isnull().sum()`, `duplicated()`, `unique()` |
| **Duplikate** | `drop_duplicates()` |
| **Texte bereinigen** | `str.strip()`, `str.title()`, `str.lower()`, `str.replace()` |
| **Zahlen bereinigen** | `pd.to_numeric()`, `str.replace()`, `astype()` |
| **Datumsangaben** | `pd.to_datetime()`, `dayfirst=True` |
| **Fehlende Werte** | `fillna()`, `dropna()` |
| **Neue Spalten** | Berechnungen, `apply()` für eigene Logik |
| **Zusammenführen** | `merge()` für DataFrames aus verschiedenen Quellen |

---
# Weitere Übungsaufgaben

Die folgenden Aufgaben beziehen sich auf die Themen dieses Notebooks. Musterlösungen findest du im Ordner `Loesungen/`.

### Aufgabe 1 - Schmutzige Produktdaten bereinigen

Bereinige den DataFrame vollständig:
- `bezeichnung`: Leerzeichen entfernen, Title Case
- `preis`: Komma → Punkt, `'N/A'` → NaN, in float umwandeln
- `lagerbestand`: in int umwandeln, fehlende Werte → 0, negative Werte → 0
- Duplikate entfernen
- Neue Spalte `gesamtwert` = Preis × Lagerbestand

> **Lösung → [`04_ETL_Transform_Aufgaben.ipynb`](04_ETL_Transform_Aufgaben.ipynb)**

### Aufgabe 2 - Fehlende Werte intelligent ergänzen

Fülle fehlende Werte kontextabhängig:
- `temperatur`: mit dem Mittelwert der jeweiligen `stadt`
- `niederschlag`: mit `0` (kein Regen gemessen)
- `beschreibung`: mit `'unbekannt'`

> **Lösung → [`04_ETL_Transform_Aufgaben.ipynb`](04_ETL_Transform_Aufgaben.ipynb)**

### Aufgabe 3 - Pivot und Rückpivot

Pivotiere den DataFrame von Long- ins Wide-Format (Zeilen = Monate, Spalten = Regionen) und füge eine `gesamt`-Spalte hinzu.
Wandle das Ergebnis danach mit `pd.melt()` wieder ins Long-Format zurück.

> **Lösung → [`04_ETL_Transform_Aufgaben.ipynb`](04_ETL_Transform_Aufgaben.ipynb)**